In [2]:
!pip install numpy pandas scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 27.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 40.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 33.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 23.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 34.9 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [seaborn]5/16 [seaborn]ib]n]


In [3]:
# Data handling
import numpy as np
import pandas as pd

# Visualization and exploratory data analysis
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Regression models and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Regression evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Notebook display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

In [6]:
df = pd.read_csv("crop_yield_dataset.csv")
print("Data loaded successfully. Here's a preview:")
df.head()
print("\nData summary:")
df.describe()

Data loaded successfully. Here's a preview:

Data summary:


,Rainfall_mm,Temperature_C,Soil_pH,Fertilizer_kg_per_ha,Pesticide_kg_per_ha,Sunlight_Hours_per_day,Farm_Size_ha,Crop_Yield_tonnes_per_ha
count,9995.000,10083.000,10091.000,9882.000,10013.000,10222.000,9996.000,10515.000
mean,910.446,24.036,6.504,149.494,5.017,7.480,5.372,8.135
std,255.016,5.037,0.810,60.756,2.452,1.518,17.083,1.451
min,100.000,5.000,3.680,0.000,0.000,2.200,0.200,4.270
25%,740.650,20.600,5.960,109.600,3.320,6.500,1.790,7.110
50%,908.800,24.000,6.500,148.900,4.960,7.500,3.280,7.960
75%,1076.500,27.400,7.050,189.275,6.690,8.500,6.062,9.040
max,3922.262,44.400,9.400,713.711,15.070,12.000,774.900,13.130


In [7]:
print("Displaying data types and non-null counts:")
df.info()

Displaying data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 10515 entries, 0 to 10514
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Soil_Type                 10184 non-null  str    
 1   Crop_Type                 10275 non-null  str    
 2   Rainfall_mm               9995 non-null   float64
 3   Temperature_C             10083 non-null  float64
 4   Soil_pH                   10091 non-null  float64
 5   Fertilizer_kg_per_ha      9882 non-null   float64
 6   Pesticide_kg_per_ha       10013 non-null  float64
 7   Sunlight_Hours_per_day    10222 non-null  float64
 8   Farm_Size_ha              9996 non-null   float64
 9   Irrigation                10204 non-null  str    
 10  Crop_Yield_tonnes_per_ha  10515 non-null  float64
dtypes: float64(8), str(3)
memory usage: 903.8 KB


In [8]:
print("\nChecking for missing values:")
print(df.isnull().sum())


Checking for missing values:
Soil_Type                   331
Crop_Type                   240
Rainfall_mm                 520
Temperature_C               432
Soil_pH                     424
Fertilizer_kg_per_ha        633
Pesticide_kg_per_ha         502
Sunlight_Hours_per_day      293
Farm_Size_ha                519
Irrigation                  311
Crop_Yield_tonnes_per_ha      0
dtype: int64


In [9]:
print("Checking for duplicate rows:")
print(df.duplicated().sum())

Checking for duplicate rows:
15


In [11]:
print("\n Checking for categorical variables:")
for col in df.columns:
    if df[col].dtype == 'str':
        print(f"{col}: {df[col].nunique()} unique values")



 Checking for categorical variables:
Soil_Type: 5 unique values
Crop_Type: 6 unique values
Irrigation: 2 unique values


In [15]:
print("\nUnique values in categorical columns:")
for col in df.columns:
    # Pandas usually stores strings as 'object' or 'string' type
    if df[col].dtype == "object" or df[col].dtype == "string":
        print()
        print(f"{col}: {df[col].nunique()} unique values -> {df[col].unique()}")



Unique values in categorical columns:

Soil_Type: 5 unique values -> <StringArray>
['Peaty', 'Clay', 'Silty', 'Sandy', 'Loamy', nan]
Length: 6, dtype: str

Crop_Type: 6 unique values -> <StringArray>
['Soybean', 'Wheat', 'Rice', 'Cotton', 'Barley', 'Maize', nan]
Length: 7, dtype: str

Irrigation: 2 unique values -> <StringArray>
['Yes', 'No', nan]
Length: 3, dtype: str


In [17]:
print("\nChecking for numerical variables:")
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        print(f"{col}: {df[col].describe()}")
        print()


Checking for numerical variables:
Rainfall_mm: count   9995.000
mean     910.446
std      255.016
min      100.000
25%      740.650
50%      908.800
75%     1076.500
max     3922.262
Name: Rainfall_mm, dtype: float64

Temperature_C: count   10083.000
mean       24.036
std         5.037
min         5.000
25%        20.600
50%        24.000
75%        27.400
max        44.400
Name: Temperature_C, dtype: float64

Soil_pH: count   10091.000
mean        6.504
std         0.810
min         3.680
25%         5.960
50%         6.500
75%         7.050
max         9.400
Name: Soil_pH, dtype: float64

Fertilizer_kg_per_ha: count   9882.000
mean     149.494
std       60.756
min        0.000
25%      109.600
50%      148.900
75%      189.275
max      713.711
Name: Fertilizer_kg_per_ha, dtype: float64

Pesticide_kg_per_ha: count   10013.000
mean        5.017
std         2.452
min         0.000
25%         3.320
50%         4.960
75%         6.690
max        15.070
Name: Pesticide_kg_per_ha, dtype: 